# IPEDS fall enrollment → `school_enrollment_annual.csv`

Builds **annual fall headcount** (IPEDS *Fall Enrollment* Part A, `ef{year}a`) for every school in `tenure_pipeline/r1_cs_departments.csv`, using `build_school_enrollment_from_ipeds.py`.

**Later:** lift cells into a `.py` module and call it from **540** CELL 0 / a small bridge cell (same `WORKSPACE_ROOT` pattern).

- **Output:** `tenure_pipeline/school_enrollment_annual.csv` + `tenure_pipeline/ipeds_unitid_crosswalk.json`
- **Cache:** `tenure_pipeline/_ipeds_cache/` (NCES zips; safe to delete to force re-download)
- **Errors / gaps:** `tenure_pipeline/ipeds_download_errors.jsonl` — one JSON object per line (404s, retries exhausted, bad zips, schema skips). **Outcome meanings + `HD_YEAR`:** see `current_documents/tenure_documents/TENURE_PIPELINE_OVERVIEW.md` **§7.5**.

In [4]:
##### EDIT BLANK ###### DISCARD
# session: 2026-04-10 — doc: TENURE_PIPELINE_OVERVIEW §7.5 (outcomes + HD_YEAR)
from functionsG_working import tyme
print(f"Edit BLANK: {tyme()}")

Edit BLANK: 04/10/2026 14:33:01


In [5]:
# === Paths (match 540 CELL 0) ===
import sys
from pathlib import Path

WORKSPACE_ROOT = Path(".").resolve()
sys.path.insert(0, str(WORKSPACE_ROOT))

TP = WORKSPACE_ROOT / "tenure_pipeline"
STAGE2_OUT = TP / "r1_cs_departments.csv"
STAGE3_ENROLLMENT = TP / "school_enrollment_annual.csv"
BUILD_SCRIPT = TP / "build_school_enrollment_from_ipeds.py"
IPEDS_CACHE = TP / "_ipeds_cache"
IPEDS_ERROR_LOG = TP / "ipeds_download_errors.jsonl"

# IPEDS fall EF: ef2000a–ef2003a lack EFALEVEL; script defaults to 2004+
YEAR_MIN = 2004
YEAR_MAX = 2024
HD_YEAR = 2023

print(f"  WORKSPACE_ROOT     : {WORKSPACE_ROOT}")
print(f"  BUILD_SCRIPT       : {BUILD_SCRIPT}")
print(f"  OUT (enrollment)   : {STAGE3_ENROLLMENT}")
print(f"  ERROR LOG (JSONL)  : {IPEDS_ERROR_LOG}")

  WORKSPACE_ROOT     : /Users/charleslevine/Library/CloudStorage/Dropbox/1-Documents/00- Dissertation/0-Next_Chapter/Code_and_Data/New SQL and PY Code/Cursor Workspace PDE
  BUILD_SCRIPT       : /Users/charleslevine/Library/CloudStorage/Dropbox/1-Documents/00- Dissertation/0-Next_Chapter/Code_and_Data/New SQL and PY Code/Cursor Workspace PDE/tenure_pipeline/build_school_enrollment_from_ipeds.py
  OUT (enrollment)   : /Users/charleslevine/Library/CloudStorage/Dropbox/1-Documents/00- Dissertation/0-Next_Chapter/Code_and_Data/New SQL and PY Code/Cursor Workspace PDE/tenure_pipeline/school_enrollment_annual.csv


In [6]:
# === Run builder (same CLI as `python tenure_pipeline/build_school_enrollment_from_ipeds.py`) ===
import subprocess

cmd = [
    sys.executable,
    str(BUILD_SCRIPT),
    "--schools-csv",
    str(STAGE2_OUT),
    "--out-csv",
    str(STAGE3_ENROLLMENT),
    "--year-min",
    str(YEAR_MIN),
    "--year-max",
    str(YEAR_MAX),
    "--hd-year",
    str(HD_YEAR),
    "--cache-dir",
    str(IPEDS_CACHE),
    "--error-log",
    str(IPEDS_ERROR_LOG),
]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, cwd=str(WORKSPACE_ROOT))
print("exit code:", proc.returncode)

Running: /opt/anaconda3/envs/tenure_net/bin/python /Users/charleslevine/Library/CloudStorage/Dropbox/1-Documents/00- Dissertation/0-Next_Chapter/Code_and_Data/New SQL and PY Code/Cursor Workspace PDE/tenure_pipeline/build_school_enrollment_from_ipeds.py --schools-csv /Users/charleslevine/Library/CloudStorage/Dropbox/1-Documents/00- Dissertation/0-Next_Chapter/Code_and_Data/New SQL and PY Code/Cursor Workspace PDE/tenure_pipeline/r1_cs_departments.csv --out-csv /Users/charleslevine/Library/CloudStorage/Dropbox/1-Documents/00- Dissertation/0-Next_Chapter/Code_and_Data/New SQL and PY Code/Cursor Workspace PDE/tenure_pipeline/school_enrollment_annual.csv --year-min 2004 --year-max 2024 --hd-year 2023 --cache-dir /Users/charleslevine/Library/CloudStorage/Dropbox/1-Documents/00- Dissertation/0-Next_Chapter/Code_and_Data/New SQL and PY Code/Cursor Workspace PDE/tenure_pipeline/_ipeds_cache
  Loading HD2023 for institution names …
  Wrote /Users/charleslevine/Library/CloudStorage/Dropbox/1-Doc

    skip: could not load (File is not a zip file)


    done in 0.7s — 168/168 schools
  Year 2012 (9/21)  loading ef2012a …
    done in 0.6s — 168/168 schools
  Year 2013 (10/21)  loading ef2013a …
    done in 0.6s — 168/168 schools
  Year 2014 (11/21)  loading ef2014a …
    done in 0.6s — 168/168 schools
  Year 2015 (12/21)  loading ef2015a …
    done in 0.6s — 168/168 schools
  Year 2016 (13/21)  loading ef2016a …
    done in 0.5s — 168/168 schools
  Year 2017 (14/21)  loading ef2017a …
    done in 0.5s — 168/168 schools
  Year 2018 (15/21)  loading ef2018a …
    done in 0.6s — 168/168 schools
  Year 2019 (16/21)  loading ef2019a …
    done in 0.5s — 168/168 schools
  Year 2020 (17/21)  loading ef2020a …
    done in 0.5s — 167/168 schools
  Year 2021 (18/21)  loading ef2021a …
    done in 0.5s — 167/168 schools
  Year 2022 (19/21)  loading ef2022a …
    done in 0.6s — 168/168 schools
  Year 2023 (20/21)  loading ef2023a …
    done in 0.6s — 168/168 schools
  Year 2024 (21/21)  loading ef2024a …
  All years finished in 12.3s
  Wrote /

    skip: could not load (404 Client Error: Not Found for url: https://nces.ed.gov/ipeds/datacenter/data/ef2024a.zip)


In [ ]:
# === Quick preview (pandas skips `#` comment lines) ===
import pandas as pd

df = pd.read_csv(STAGE3_ENROLLMENT, comment="#")
print(len(df), "rows")
display(df.head(12))
df.groupby("year").size().tail(8)